In [9]:
from transformers import pipeline
import torch

# Check if GPU available (speeds things up)
device = 0 if torch.cuda.is_available() else -1
print(f"Using: {'GPU ✅' if device == 0 else 'CPU (slower but works fine)'}")

# Load DeBERTa NLI model
# First run downloads ~180MB — cached after that
print("\nLoading DeBERTa NLI model...")
nli_model = pipeline(
    "text-classification",
    model="cross-encoder/nli-deberta-v3-base",
    device=device
)
print("NLI Model loaded ✅")

Using: CPU (slower but works fine)

Loading DeBERTa NLI model...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NLI Model loaded ✅


In [10]:
def check_entailment(claim: str, span: str) -> dict:
    """
    Checks if a document span supports a claim.
    
    IMPORTANT: NLI format is always:
        PREMISE    = span (what the document says)
        HYPOTHESIS = claim (what the answer claims)
    Never reverse these.
    
    Returns:
        {
            "label": "ENTAILMENT" | "NEUTRAL" | "CONTRADICTION",
            "confidence": 0.95,
            "claim": "...",
            "span": "..."
        }
    """
    
    # NLI input format: "premise [SEP] hypothesis"
    nli_input = f"{span} [SEP] {claim}"
    
    result = nli_model(nli_input)
    
    label = result[0]["label"].upper()
    confidence = round(result[0]["score"], 4)
    
    # Normalize label names across models
    label_map = {
        "ENTAILMENT": "ENTAILMENT",
        "NEUTRAL":    "NEUTRAL",
        "CONTRADICTION": "CONTRADICTION",
        "LABEL_0": "CONTRADICTION",
        "LABEL_1": "NEUTRAL", 
        "LABEL_2": "ENTAILMENT"
    }
    
    normalized_label = label_map.get(label, label)
    
    return {
        "label": normalized_label,
        "confidence": confidence,
        "claim": claim,
        "span": span
    }

In [11]:
# Test obvious cases first to confirm model is working

test_pairs = [
    {
        "claim": "The Eiffel Tower stands 330 metres tall",
        "span": "It stands 330 metres tall and is one of the most recognizable structures in the world.",
        "expected": "ENTAILMENT"
    },
    {
        "claim": "The Eiffel Tower was built in 1920",
        "span": "It was constructed between 1887 and 1889 as the centerpiece of the 1889 World's Fair.",
        "expected": "CONTRADICTION"
    },
    {
        "claim": "The Eiffel Tower has a restaurant on the top floor",
        "span": "The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars in Paris, France.",
        "expected": "NEUTRAL"
    }
]

print("BASIC ENTAILMENT TESTS")
print("="*60)

for pair in test_pairs:
    result = check_entailment(pair["claim"], pair["span"])
    status = "✅" if result["label"] == pair["expected"] else "❌"
    
    print(f"\n{status} EXPECTED: {pair['expected']}")
    print(f"   GOT:      {result['label']} (confidence: {result['confidence']})")
    print(f"   CLAIM:    {pair['claim']}")
    print(f"   SPAN:     {pair['span'][:80]}...")

BASIC ENTAILMENT TESTS

❌ EXPECTED: ENTAILMENT
   GOT:      NEUTRAL (confidence: 0.9986)
   CLAIM:    The Eiffel Tower stands 330 metres tall
   SPAN:     It stands 330 metres tall and is one of the most recognizable structures in the ...

✅ EXPECTED: CONTRADICTION
   GOT:      CONTRADICTION (confidence: 0.9998)
   CLAIM:    The Eiffel Tower was built in 1920
   SPAN:     It was constructed between 1887 and 1889 as the centerpiece of the 1889 World's ...

✅ EXPECTED: NEUTRAL
   GOT:      NEUTRAL (confidence: 0.9996)
   CLAIM:    The Eiffel Tower has a restaurant on the top floor
   SPAN:     The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars in...


In [12]:
def rerank_with_nli(claim: str, top_3_spans: list[dict]) -> dict:
    """
    Updated priority:
    1. CONTRADICTION — surface these first (strongest signal)
    2. ENTAILMENT    — confirm supported claims
    3. NEUTRAL       — last resort
    
    This ensures contradictions are never buried.
    """
    
    all_results = []
    
    for rank, span_dict in enumerate(top_3_spans):
        nli_result = check_entailment(claim, span_dict["sentence"])
        all_results.append({
            "rank": rank + 1,
            "span": span_dict["sentence"],
            "doc_index": span_dict["doc_index"],
            "retrieval_score": span_dict["score"],
            "nli_label": nli_result["label"],
            "nli_confidence": nli_result["confidence"]
        })
    
    # Priority 1: Surface contradictions immediately
    contradictions = [r for r in all_results if r["nli_label"] == "CONTRADICTION"]
    if contradictions:
        best = max(contradictions, key=lambda x: x["nli_confidence"])
    
    # Priority 2: Entailment
    else:
        entailments = [r for r in all_results if r["nli_label"] == "ENTAILMENT"]
        if entailments:
            best = max(entailments, key=lambda x: x["nli_confidence"])
        
        # Priority 3: Neutral — nothing useful found
        else:
            best = max(all_results, key=lambda x: x["nli_confidence"])
    
    return {
        "best_span": best["span"],
        "doc_index": best["doc_index"],
        "label": best["nli_label"],
        "confidence": best["nli_confidence"],
        "retrieval_rank": best["rank"],
        "reranked": best["rank"] != 1,
        "all_results": all_results
    }

In [13]:
import nltk
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

source_docs = [
    """The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars 
    in Paris, France. It was constructed between 1887 and 1889 as the centerpiece 
    of the 1889 World's Fair. The tower was designed and built by Alexandre Gustave Eiffel, 
    a French civil engineer. It stands 330 metres tall and is one of the most recognizable 
    structures in the world. The tower receives approximately 7 million visitors per year.""",

    """Photosynthesis is a biological process by which green plants and some other organisms 
    convert light energy into chemical energy stored in glucose. The process takes place 
    primarily in the chloroplasts of plant cells, using chlorophyll to absorb sunlight. 
    Carbon dioxide from the air and water from the soil are the raw materials. 
    The byproducts of photosynthesis include oxygen, which is released into the atmosphere. 
    This process is fundamental to life on Earth.""",

    """Python is a high-level, general-purpose programming language created by Guido van Rossum. 
    The first version was released in 1991. Python emphasizes code readability and simplicity, 
    making it accessible to beginners and professionals alike. It has become one of the most 
    popular languages for data science, machine learning, and artificial intelligence applications. 
    Python supports multiple programming paradigms including procedural, object-oriented, 
    and functional programming."""
]

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def split_into_sentences(docs):
    all_sentences = []
    for doc_idx, doc in enumerate(docs):
        sentences = nltk.sent_tokenize(doc.strip())
        for sent in sentences:
            sent = sent.strip()
            if len(sent) > 10:
                all_sentences.append({"sentence": sent, "doc_index": doc_idx})
    return all_sentences

def build_faiss_index(sentences):
    texts = [s["sentence"] for s in sentences]
    embeddings = embed_model.encode(texts)
    embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.astype('float32'))
    return index, sentences

# ← top_k increased to 5
def retrieve_span(claim, index, sentence_store, top_k=5):
    claim_emb = embed_model.encode([claim])
    claim_emb = claim_emb / np.linalg.norm(claim_emb, axis=1, keepdims=True)
    scores, indices = index.search(claim_emb.astype('float32'), top_k)
    best = sentence_store[indices[0][0]]
    return {
        "sentence": best["sentence"],
        "doc_index": best["doc_index"],
        "similarity_score": round(float(scores[0][0]), 4),
        "low_confidence": float(scores[0][0]) < 0.70,
        "top_3_spans": [
            {
                "sentence": sentence_store[indices[0][i]]["sentence"],
                "doc_index": sentence_store[indices[0][i]]["doc_index"],
                "score": round(float(scores[0][i]), 4)
            }
            for i in range(top_k)
        ]
    }

sentences = split_into_sentences(source_docs)
faiss_index, sentence_store = build_faiss_index(sentences)
print(f"Index built ✅  |  Total sentences: {len(sentences)}")

# Verify the problem — check what top 5 looks like for the failing claim
print("\nDEBUG — Top 5 spans for 'built in 1950':")
debug = retrieve_span("The Eiffel Tower was built in 1950", faiss_index, sentence_store)
for i, s in enumerate(debug["top_3_spans"]):
    print(f"  Rank {i+1} [{s['score']}]: {s['sentence'][:80]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index built ✅  |  Total sentences: 15

DEBUG — Top 5 spans for 'built in 1950':
  Rank 1 [0.7534]: The tower was designed and built by Alexandre Gustave Eiffel, 
    a French civi
  Rank 2 [0.6817]: The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars 
 
  Rank 3 [0.3293]: The tower receives approximately 7 million visitors per year.
  Rank 4 [0.2617]: It stands 330 metres tall and is one of the most recognizable 
    structures in
  Rank 5 [0.244]: It was constructed between 1887 and 1889 as the centerpiece 
    of the 1889 Wor


In [14]:
# Day 1 claims — exact output from our claim extractor
all_claims = [
    "The Eiffel Tower was built in 1889",
    "The Eiffel Tower stands 330 metres tall",
    "The Eiffel Tower was designed by Gustave Eiffel",
    "The Eiffel Tower is located in Paris, France",
    "Photosynthesis occurs in the chloroplasts",
    "Oxygen is released as a byproduct of photosynthesis",
    "Python was created by Guido van Rossum",
    "Python was first released in 1991",
    "Python is widely used in data science"
]

print("FULL PIPELINE: CLAIM → RETRIEVAL → NLI RERANKING")
print("="*60)

label_emoji = {
    "ENTAILMENT":    "✅",
    "NEUTRAL":       "⚪",
    "CONTRADICTION": "❌"
}

for claim in all_claims:
    # Step 1: Retrieve top 3 spans
    retrieval = retrieve_span(claim, faiss_index, sentence_store)
    
    # Step 2: NLI reranking across top 3
    result = rerank_with_nli(claim, retrieval["top_3_spans"])
    
    emoji = label_emoji[result["label"]]
    reranked_note = "🔄 RERANKED" if result["reranked"] else ""
    
    print(f"\n{emoji} CLAIM:      {claim}")
    print(f"   SPAN:       {result['best_span'][:90]}...")
    print(f"   VERDICT:    {result['label']} ({result['confidence']}) {reranked_note}")
    print(f"   USED RANK:  {result['retrieval_rank']} of 3")

FULL PIPELINE: CLAIM → RETRIEVAL → NLI RERANKING

⚪ CLAIM:      The Eiffel Tower was built in 1889
   SPAN:       The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars 
    in Pari...
   VERDICT:    NEUTRAL (0.9997) 🔄 RERANKED
   USED RANK:  2 of 3

⚪ CLAIM:      The Eiffel Tower stands 330 metres tall
   SPAN:       The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars 
    in Pari...
   VERDICT:    NEUTRAL (0.9998) 🔄 RERANKED
   USED RANK:  2 of 3

✅ CLAIM:      The Eiffel Tower was designed by Gustave Eiffel
   SPAN:       The tower was designed and built by Alexandre Gustave Eiffel, 
    a French civil engineer...
   VERDICT:    ENTAILMENT (0.9838) 
   USED RANK:  1 of 3

✅ CLAIM:      The Eiffel Tower is located in Paris, France
   SPAN:       The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars 
    in Pari...
   VERDICT:    ENTAILMENT (0.9931) 
   USED RANK:  1 of 3

✅ CLAIM:      Photosynthesis occurs in t

In [15]:
# These are WRONG claims — system must catch them all

wrong_claims = [
    {
        "claim": "The Eiffel Tower was built in 1950",
        "expected": "CONTRADICTION"
    },
    {
        "claim": "Photosynthesis releases carbon dioxide as a byproduct",
        "expected": "CONTRADICTION"
    },
    {
        "claim": "Python was created by Linus Torvalds",
        "expected": "CONTRADICTION"
    },
    {
        "claim": "The Eiffel Tower is located in London",
        "expected": "CONTRADICTION"
    }
]

print("CONTRADICTION DETECTION TEST")
print("="*60)
print("These claims are WRONG — system must catch them all\n")

caught = 0
for item in wrong_claims:
    retrieval = retrieve_span(item["claim"], faiss_index, sentence_store)
    result = rerank_with_nli(item["claim"], retrieval["top_3_spans"])
    
    detected = result["label"] == "CONTRADICTION"
    status = "✅ CAUGHT" if detected else "❌ MISSED"
    if detected:
        caught += 1
    
    print(f"{status}: {item['claim']}")
    print(f"         Got: {result['label']} ({result['confidence']})")
    print(f"         Span: {result['best_span'][:80]}...\n")

print(f"Contradiction Detection Rate: {caught}/{len(wrong_claims)}")

CONTRADICTION DETECTION TEST
These claims are WRONG — system must catch them all

✅ CAUGHT: The Eiffel Tower was built in 1950
         Got: CONTRADICTION (0.9998)
         Span: It was constructed between 1887 and 1889 as the centerpiece 
    of the 1889 Wor...

✅ CAUGHT: Photosynthesis releases carbon dioxide as a byproduct
         Got: CONTRADICTION (0.9996)
         Span: The byproducts of photosynthesis include oxygen, which is released into the atmo...

✅ CAUGHT: Python was created by Linus Torvalds
         Got: CONTRADICTION (0.9999)
         Span: Python is a high-level, general-purpose programming language created by Guido va...

✅ CAUGHT: The Eiffel Tower is located in London
         Got: CONTRADICTION (0.9998)
         Span: The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars 
 ...

Contradiction Detection Rate: 4/4


In [16]:
print("""
✅ Day 3 Complete!

Copy these functions into verifaith/entailment_checker.py:
  → check_entailment()
  → rerank_with_nli()

What we have now:
  ✅ Day 1 → claim_extractor.py     (Groq LLaMA)
  ✅ Day 2 → span_retriever.py      (FAISS + MiniLM)
  ✅ Day 3 → entailment_checker.py  (DeBERTa NLI)

Day 4: Wire all 3 modules into scorer.py
       Input  → answer + source docs
       Output → faithfulness score + full report
""")


✅ Day 3 Complete!

Copy these functions into verifaith/entailment_checker.py:
  → check_entailment()
  → rerank_with_nli()

What we have now:
  ✅ Day 1 → claim_extractor.py     (Groq LLaMA)
  ✅ Day 2 → span_retriever.py      (FAISS + MiniLM)
  ✅ Day 3 → entailment_checker.py  (DeBERTa NLI)

Day 4: Wire all 3 modules into scorer.py
       Input  → answer + source docs
       Output → faithfulness score + full report

